# Project 2 - Task 2.2 Notebook


In [106]:
import cv2
import sns
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import warnings
import torch
from ultralytics import YOLO
from transformers import pipeline
from torchvision.datasets import VOCDetection
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    RTDetrV2ForObjectDetection,
    RTDetrForObjectDetection, RTDetrImageProcessor,
    DetrImageProcessor,
    DetrForObjectDetection
)
import pandas as pd
from taco_dataset import TACODETRDetectionDataset
import os

In [107]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [108]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device", device)

Using device cuda:0


In [109]:
# Mapping between class IDs and labels

label2id = {'Plastic bag & wrapper': 0,
  'Cigarette': 1,
  'Bottle': 2,
  'Bottle cap': 3,
  'Can': 4,
  'Carton': 5}

id2label = {v: k for k, v in label2id.items()}
label2id, id2label
categories = list(id2label.values())

image_size = 480
checkpoint_rtdetr = "PekingU/rtdetr_r50vd"

In [110]:
# Instantiate the image processor
image_processor = AutoImageProcessor.from_pretrained(
    checkpoint_rtdetr,
    do_resize=True,     # Resize the images to the expected size
    size={"width": image_size, "height": image_size},
    use_fast=True,      # Use the fast version of the processor
)

In [111]:
train_dataset = TACODETRDetectionDataset(
    img_folder="/home/jb/Desktop/Projects/CV/cv-project2/taco",
    ann_file="/home/jb/Desktop/Projects/CV/cv-project2/taco/annotations_train.json",
    processor=image_processor,
)

validation_dataset = TACODETRDetectionDataset(
    img_folder="/home/jb/Desktop/Projects/CV/cv-project2/taco",
    ann_file="/home/jb/Desktop/Projects/CV/cv-project2/taco/annotations_val.json",
    processor=image_processor,
)

test_dataset = TACODETRDetectionDataset(
    img_folder="/home/jb/Desktop/Projects/CV/cv-project2/taco",
    ann_file="/home/jb/Desktop/Projects/CV/cv-project2/taco/annotations_test.json",
    processor=image_processor,
)

loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


In [ ]:
targets_list = []
test_dataset = test_dataset.coco_dataset

for i in range(len(test_dataset)):
    img, target = test_dataset[i]

    boxes = []
    labels = []
    for obj in target:
        b = obj['bbox'] 
        boxes.append([b[0], b[1], b[0] + b[2], b[1] + b[3]])

    targets_list.append({
        "boxes": torch.tensor(boxes, dtype=torch.float32).to(device),
        "labels": torch.zeros(len(boxes), dtype=torch.int64).to(device)
    })

In [ ]:
def topk_strategy(boxes, scores, k, device):

    sort_ind = torch.argsort(scores, descending=True)
    k_best = min(k, len(sort_ind))
    top = sort_ind[:k_best]

    final_boxes = boxes[top]
    final_scores = scores[top]
    final_labels = torch.zeros(k_best, dtype=torch.int64, device=device)

    return {
        "boxes": final_boxes,
        "scores": final_scores,
        "labels": final_labels
    }

def convert_yolo_topk(yolo_results, k, device):
    boxes = yolo_results.boxes.xyxy
    scores = yolo_results.boxes.conf
    return topk_strategy(boxes, scores, k, device)

def convert_detr_topk(results, k, device):
    boxes = results["boxes"]
    scores = results["scores"]
    return topk_strategy(boxes, scores, k, device)


In [114]:
def evaluate_yolo_topk(yolo_model, subset, targets_list, conf):
    metric = MeanAveragePrecision()

    for sample, target in zip(subset, targets_list):
        img = sample[0]
        img_np = np.array(img)
        k = len(target["labels"])
        yolo_raw = yolo_model(img_np, conf=conf, verbose=False)[0]
        preds = convert_yolo_topk(yolo_raw, k, yolo_model.device)
        metric.update([preds], [target])

    return metric.compute()

def evaluate_detr_topk(processor, model, subset, targets_list, conf, device):
    metric = MeanAveragePrecision()

    for sample, target in zip(subset, targets_list):
        img = sample[0]
        img_np = np.array(img)
        k = len(target["labels"])
        inputs = processor(images=img_np, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        target_sizes = torch.tensor([img.size[::-1]]).to(device)
        results = processor.post_process_object_detection(
            outputs, target_sizes=target_sizes, threshold=conf
        )[0]

        preds = convert_detr_topk(results, k, device)
        metric.update([preds], [target])

    return metric.compute()

In [ ]:
os.makedirs('results/task2_2', exist_ok=True)

experiments = [
    ('yolo', 'yolov8n.pt'),
    ('yolo', 'yolov8m.pt'),
    ('yolo', 'yolov8x.pt'),
    ('yolo', 'yolo11n.pt'),
    ('yolo', 'yolo11m.pt'),
    ('yolo', 'yolo11x.pt'),
    ('detr', 'facebook/detr-resnet-50'),
    ('detr', 'facebook/detr-resnet-101'),
    ('rtdetr', 'PekingU/rtdetr_r50vd'),
    ('rtdetr', 'PekingU/rtdetr_r101vd'),
    ('rtdetr-v2', "PekingU/rtdetr_v2_r50vd"),
    ('rtdetr-v2', "PekingU/rtdetr_v2_r101vd"),
]

confidence_thresholds = [0.001, 0.1, 0.2, 0.3]

results = []

for conf in confidence_thresholds:
    for m_type, m_name in experiments:

        if m_type == 'yolo':
            model = YOLO(m_name).to(device)
            result = evaluate_yolo_topk(model, test_dataset, targets_list, conf=conf)
        elif m_type == 'detr':
            proc = DetrImageProcessor.from_pretrained(m_name)
            model = DetrForObjectDetection.from_pretrained(m_name).to(device)
            result = evaluate_detr_topk(proc, model, test_dataset, targets_list, conf=conf, device=device)
        elif m_type == 'rt-detr':
            proc = RTDetrImageProcessor.from_pretrained(m_name)
            model = RTDetrForObjectDetection.from_pretrained(m_name).to(device)
            result = evaluate_detr_topk(proc, model, test_dataset, targets_list, conf=conf, device=device)
        else: # rtdetr-v2
            proc = RTDetrImageProcessor.from_pretrained(m_name)
            model = RTDetrV2ForObjectDetection.from_pretrained(m_name).to(device)
            result = evaluate_detr_topk(proc, model, test_dataset, targets_list, conf=conf, device=device)

        results.append({
                "Model": m_name,
                "Conf_Threshold": conf,
                "mAP": result['map'].item(),
                "mAP_50": result['map_50'].item(),
                "mAP_75": result['map_75'].item()
            })

df_results = pd.DataFrame(results)
df_results.sort_values(by=["mAP"], ascending=False, inplace=True)

path = os.path.join('results/task2_2', "results.csv")
df_results.to_csv(path, index=False)

df_results



--- Testing Confidence Threshold: 0.001 ---


KeyboardInterrupt: 